<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h1>مرحلهٔ دوم: پایگاه داده و کارایی</h1>

<p>آماده‌سازی، بررسی و پاک‌سازی داده‌های Olist با پکیج مشترک تکمیل شده‌اند. اتصال واقعی و اجرای دو فایل DDL تکمیل شده‌اند؛ بارگذاری داده‌ها در PostgreSQL و ارزیابی کوئری‌ها هنوز انجام نشده‌اند.</p>

<p>هدف مرحله، توسعهٔ همان پکیج کارخانه برای دریافت داده‌های <bdi dir="ltr">Olist</bdi> و بارگذاری آن‌ها همراه خروجی مرحلهٔ اول در <bdi dir="ltr">PostgreSQL</bdi> است. بودجهٔ هر کوئری ۲۰۰ میلی‌ثانیه و سقف اجرای نوت‌بوک، بدون بارگذاری اولیه، ۱۰ دقیقه است.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۱. آماده‌سازی پروژه</h2>

<p>سلول بعد ریشهٔ پروژه را از پوشهٔ فعلی یا والدهای آن پیدا می‌کند، مسیر کد مشترک را در دسترس قرار می‌دهد و <bdi dir="ltr">seed</bdi> را تعریف می‌کند. این سلول هیچ اتصال یا تغییری در دیتابیس ایجاد نمی‌کند.</p>

</div>


In [2]:
import json
import os
import random
import sys
from pathlib import Path

start_dir = Path.cwd().resolve()
candidates = [start_dir, *start_dir.parents, start_dir / 'machine_learning_challenge']
PROJECT_ROOT = next((path for path in candidates
                     if (path / 'pyproject.toml').is_file()
                     and (path / 'src/data_factory').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the notebook within the project or its parent directory.')
os.chdir(PROJECT_ROOT)
source_dir = str(PROJECT_ROOT / 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

import data_factory

settings_example = json.loads(Path('config/database.example.json').read_text(encoding='utf-8'))
SEED = settings_example['benchmark']['seed']
random.seed(SEED)
QUERY_BUDGET_MS = settings_example['benchmark']['query_budget_ms']
print('Status: project setup ready.')
print('Shared package:', data_factory.__name__)
print('Seed:', SEED)
print('Query budget (ms):', QUERY_BUDGET_MS)
print('Database connections opened: 0')

Status: project setup ready.
Shared package: data_factory
Seed: 42
Query budget (ms): 200
Database connections opened: 0


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۱.۱. تنظیم اتصال واقعی Python به PostgreSQL</h2>
<p>تنظیمات واقعی اتصال در فایل محلی <code dir="ltr">.env</code> در ریشهٔ پروژه قرار دارند. میزبان <code dir="ltr">localhost</code>، پورت تأییدشدهٔ کاربر <code dir="ltr">5432</code>، کاربر <code dir="ltr">postgres</code> و دیتابیس انتخاب‌شده <code dir="ltr">ds_learning</code> است. رمز فقط در فایل محلی نگهداری می‌شود و این فایل از Git خارج است.</p>
<p><code dir="ltr">load_database_settings()</code> فایل ریشهٔ پروژه را مستقل از مسیر اجرای نوت‌بوک می‌خواند. مقدارهای موجود در محیط فرایند بر فایل اولویت دارند. بارگذاری فایل، محیط فرایند را تغییر نمی‌دهد. برای اطلاعات حساس از نمایش مستقیم URL یا شیء Engine استفاده نمی‌کنیم.</p>
<p>Engine در این بخش، پیش از طراحی و اجرای schema و SQL ساخته می‌شود و برای بخش‌های بعدی در متغیر <code dir="ltr">engine</code> باقی می‌ماند. سلول زیر از تنظیمات واقعی خوانده‌شده از فایل استفاده می‌کند. پس از ساخت Engine، <code dir="ltr">connect()</code> اتصال واقعی را دریافت می‌کند و یک SELECT خواندنی، صحت اتصال، نام دیتابیس، کاربر و نسخهٔ سرور را گزارش می‌کند. رمز در کد یا خروجی این سلول نیست.</p>

</div>


In [3]:
from sqlalchemy import text
from data_factory.database.connection import load_database_settings, create_database_engine

# Re-running this cell releases the previous pool before replacing the engine.
if "engine" in globals():
    engine.dispose()

db_settings = load_database_settings()
engine = create_database_engine(db_settings)
with engine.connect() as connection:
    result = connection.execute(text(
        "SELECT 1 AS connection_ok, current_database() AS database_name, "
        "current_user AS database_user, current_setting('server_version') AS server_version"
    )).mappings().one()
    print("Target:", db_settings.safe_url)
    print("SELECT 1:", result["connection_ok"])
    print("Database:", result["database_name"])
    print("User:", result["database_user"])
    print("PostgreSQL version:", result["server_version"])

# Reuse engine in later SQL cells; each with block releases its Connection.


Target: postgresql+psycopg://localhost:5432/ds_learning
SELECT 1: 1
Database: ds_learning
User: postgres
PostgreSQL version: 18.6


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3>استفاده از اتصال در ادامهٔ نوت‌بوک</h3>
<p>متغیر <code dir="ltr">engine</code> از ابتدای نوت‌بوک در دسترس بخش‌های SQL است. برای هر واحد کار، اتصال را در یک بلوک <code dir="ltr">with</code> می‌گیریم و در پایان همان بلوک به pool برمی‌گردانیم. یک Connection را در تمام سلول‌ها و هنگام توقف کار باز نگه نمی‌داریم.</p>
<p>اجرای فایل‌های DDL، ایجاد schema و جدول، بارگذاری داده‌ها و کوئری‌ها از همین Engine و از طریق SQLAlchemy انجام می‌شود. عملیات نوشتن در بلوک <code dir="ltr">engine.begin()</code> تراکنش دارند؛ خواندن نتایج با <code dir="ltr">engine.connect()</code> انجام می‌شود. تعریف‌های SQL در فایل‌های <code dir="ltr">sql/</code> باقی می‌مانند.</p>
<p>آزادسازی pool با <code dir="ltr">engine.dispose()</code> به آخر نوت‌بوک منتقل شده است تا Engine برای بخش‌های بعدی قابل استفاده باشد. سلول‌های آماده‌سازی، اتصال، اجرای DDL و بررسی ساختار، و آزادسازی در یک فرایند تازهٔ IPython اجرا شدند؛ اجرای کامل نوت‌بوک یا Jupyter kernel انجام نشده است. سلول‌ها و خروجی‌های بررسی و پاک‌سازی داده‌ها حفظ شده‌اند. DDL در بخش ۳.۱ اجرا شده است و بارگذاری داده‌ها هنوز آغاز نشده است.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۲. منابع داده و طراحی <bdi dir="ltr">schema</bdi></h2>

<p>در این بخش، نه فایل <bdi dir="ltr">Olist</bdi> و خروجی معتبر مرحلهٔ اول بررسی خواهند شد. نوع ستون‌ها، کلیدها، محدودیت‌ها و رفتار رکوردهای ناقض یکپارچگی باید با مشاهدهٔ داده تعیین شوند. داده‌های <bdi dir="ltr">Olist</bdi> و <bdi dir="ltr">NYC</bdi> 311 در دو <bdi dir="ltr">schema</bdi> جداگانهٔ یک دیتابیس قرار خواهند گرفت. تعریف‌ها در <code dir="ltr" style="unicode-bidi: isolate; display: inline-block;">sql/ddl/</code> ثبت می‌شوند.</p>

<p>وضعیت: بررسی و پاک‌سازی Olist و بررسی خروجی NYC 311 انجام شده‌اند؛ DDL و بارگذاری باقی مانده‌اند.</p>

</div>


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


orders_path = "data/stage_02/raw/olist_orders_dataset.csv"
orders = pd.read_csv(orders_path, dtype={'order_id': 'string', 'customer_id': 'string'})
display(orders)
display(orders.isnull().sum())
display(orders.duplicated('order_id').sum()) # Candidate primary key; also inspect missing values.
display(orders.duplicated('customer_id').sum()) # This counts repetition; parent existence is checked below.

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

np.int64(0)

np.int64(0)

In [5]:
customers_path = "data/stage_02/raw/olist_customers_dataset.csv"
customers = pd.read_csv(customers_path, dtype={'customer_id': 'string', 'customer_unique_id': 'string', 'customer_zip_code_prefix': 'string'})
display(customers)
display(customers.isnull().sum())
display(customers.duplicated('customer_id').sum())
display(customers.duplicated('customer_unique_id').sum())

unknown_customers = (orders['customer_id'].notna() & ~orders['customer_id'].isin(customers['customer_id']))
display(unknown_customers.sum())

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,03937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,06764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

np.int64(0)

np.int64(3345)

np.int64(0)

In [6]:
geolocation_path = "data/stage_02/raw/olist_geolocation_dataset.csv"
geolocation = pd.read_csv(geolocation_path, dtype={'geolocation_zip_code_prefix': 'string'})
display(geolocation)
display(geolocation.isnull().sum())


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [7]:
order_items_path = "data/stage_02/raw/olist_order_items_dataset.csv"
order_items = pd.read_csv(order_items_path, dtype={'order_id': 'string', 'product_id': 'string', 'seller_id': 'string', 'price': 'string', 'freight_value': 'string'})
display(order_items)
display(order_items.isnull().sum())
display(order_items.duplicated(['order_id']).sum())
display(order_items.duplicated(['order_id','order_item_id']).sum()) # One composite primary key: (order_id, order_item_id).


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

np.int64(13984)

np.int64(0)

In [8]:
order_payments_path = "data/stage_02/raw/olist_order_payments_dataset.csv"
order_payments = pd.read_csv(order_payments_path, dtype={'order_id': 'string', 'payment_value': 'string'})
display(order_payments)
display(order_payments.isnull().sum())


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [9]:
order_review_path = "data/stage_02/raw/olist_order_reviews_dataset.csv"
order_review = pd.read_csv(order_review_path, dtype={'review_id': 'string', 'order_id': 'string'})
display(order_review)
display(order_review.isnull().sum())
display(order_review.duplicated(['review_id']).sum()) # not a primary key
display(order_review.duplicated(['review_id','order_id']).sum()) #  primary key


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

np.int64(814)

np.int64(0)

In [10]:
product_path = "data/stage_02/raw/olist_products_dataset.csv"
product = pd.read_csv(product_path, dtype={'product_id': 'string'})
display(product)
display(product.isnull().sum())
display(product.duplicated(['product_id']).sum())

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

np.int64(0)

In [11]:
seller_path = "data/stage_02/raw/olist_sellers_dataset.csv"
seller = pd.read_csv(seller_path, dtype={'seller_id': 'string', 'seller_zip_code_prefix': 'string'})
display(seller)
display(seller.isnull().sum())
display(seller.duplicated(['seller_id']).sum()) # it's primary key

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,04650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

np.int64(0)

In [12]:
product_category_path = "data/stage_02/raw/product_category_name_translation.csv"
product_category = pd.read_csv(product_category_path)
display(product_category)
display(product_category.isnull().sum())


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


product_category_name            0
product_category_name_english    0
dtype: int64

<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="schema-sources">۲.۱. خلاصهٔ منابع و مقدارهای خالی</h3>

<p>منطق بررسی‌های این بخش در <code dir="ltr">src/data_factory/database/source_inspection.py</code> قرار دارد؛ نوت‌بوک توابع را فراخوانی و نتیجه‌ها را نمایش می‌دهد. شناسه، کد پستی و مبلغ هنگام خواندن به‌صورت متن حفظ می‌شوند؛ تبدیل آزمایشی نوع فقط برای بررسی است و فایل خام تغییر نمی‌کند.</p>

<p>تعداد مقدار null با رشتهٔ خالی یا فقط فاصله یکسان نیست. جدول دوم هر دو را جداگانه و مجموع آن‌ها را نشان می‌دهد. سطر تکراری کامل نیز با تکرار کلید اصلی تفاوت دارد.</p>

</div>


In [13]:
from data_factory.database.source_inspection import inspect_olist, save_evidence

frames = {
    "orders": orders, "customers": customers, "geolocation": geolocation,
    "order_items": order_items, "order_payments": order_payments,
    "order_reviews": order_review, "products": product, "sellers": seller,
    "category_translation": product_category,
}
inspection = inspect_olist(PROJECT_ROOT, frames) 
display(inspection["sources"])
display(inspection["missing"].query("effective_missing_rows > 0"))


,table,rows,columns,exact_duplicate_extra_rows
0,orders,99441,8,0
1,customers,99441,5,0
2,geolocation,1000163,5,261831
3,order_items,112650,7,0
4,order_payments,103886,5,0
5,order_reviews,99224,7,0
6,products,32951,9,0
7,sellers,3095,4,0
8,category_translation,71,2,0


,table,column,null_rows,blank_rows,effective_missing_rows
4,orders,order_approved_at,160,0,160
5,orders,order_delivered_carrier_date,1783,0,1783
6,orders,order_delivered_customer_date,2965,0,2965
33,order_reviews,review_comment_title,87656,2,87658
34,order_reviews,review_comment_message,58247,27,58274
38,products,product_category_name,610,0,610
39,products,product_name_lenght,610,0,610
40,products,product_description_lenght,610,0,610
41,products,product_photos_qty,610,0,610
42,products,product_weight_g,2,0,2


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="schema-keys">۲.۲. کلیدهای ساده و مرکب</h3>

<p>کلید اصلی باید هم غیرخالی و هم یکتا باشد. در کلید مرکب، ترکیب ستون‌ها معیار یکتایی است؛ دو ستون، دو کلید اصلی مستقل نیستند. جدول زیر کلید پرداخت‌ها و نامزدهای باقی‌مانده را هم بررسی می‌کند.</p>

<p>تکرار شناسهٔ ثابت مشتری می‌تواند ناشی از خرید مجدد باشد. شناسهٔ نظر و پیش‌شمارهٔ جغرافیایی نیز بدون آزمون نباید UNIQUE شوند. تعداد سطر اضافهٔ تکراری در جدول پس از اولین نمونهٔ هر کلید شمرده می‌شود.</p>

</div>


In [14]:
display(inspection["keys"])


,table,key,missing_rows,duplicate_extra_rows,valid_candidate
0,orders,order_id,0,0,True
1,customers,customer_id,0,0,True
2,customers,customer_unique_id,0,3345,False
3,order_items,order_id + order_item_id,0,0,True
4,order_payments,order_id + payment_sequential,0,0,True
5,order_reviews,review_id,0,814,False
6,order_reviews,order_id,0,551,False
7,order_reviews,review_id + order_id,0,0,True
8,products,product_id,0,0,True
9,sellers,seller_id,0,0,True


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="schema-relationships">۲.۳. کنترل کلیدهای خارجی</h3>

<p>تکراری نبودن شناسهٔ فرزند، وجود آن در جدول والد را ثابت نمی‌کند. با <code dir="ltr">isin</code> وجود هر شناسهٔ غیرخالی در مجموعهٔ کلیدهای والد بررسی می‌شود. مقدار خالی جداگانه شمرده شده است.</p>

<p>شناسهٔ ناموجود در فایل ترجمه را با حذف محصول حل نمی‌کنیم: جدول مرجع دسته‌ها از اجتماع دسته‌های محصولات و ترجمه ساخته خواهد شد و ترجمهٔ انگلیسی می‌تواند خالی بماند. این بررسی فعلاً هیچ ردیفی را حذف یا قرنطینه نمی‌کند.</p>

</div>


In [15]:
display(inspection["foreign_keys"])
print("Proposed product_categories rows:", inspection["proposed_category_rows"])


,child,column,parent,missing_child_rows,unmatched_rows,unmatched_distinct,unmatched_values
0,orders,customer_id,customers,0,0,0,[]
1,order_items,order_id,orders,0,0,0,[]
2,order_items,product_id,products,0,0,0,[]
3,order_items,seller_id,sellers,0,0,0,[]
4,order_payments,order_id,orders,0,0,0,[]
5,order_reviews,order_id,orders,0,0,0,[]
6,products,product_category_name,category_translation,610,13,2,"[pc_gamer, portateis_cozinha_e_preparadores_de..."


Proposed product_categories rows: 73


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="schema-types">۲.۴. نوع ستون‌ها و قواعد عددی</h3>

<p>جدول نوع پیشنهادی از قرارداد JSON خوانده می‌شود؛ فایل JSON تنظیمات طراحی است و ماژول بررسی آن را برای کنترل قواعد می‌خواند. این نوع‌ها هنوز در PostgreSQL ساخته نشده‌اند.</p>

<p>مبلغ با Decimal و دقت پیشنهادی NUMERIC(12,2) کنترل می‌شود تا اعشار نامعتبر با گردکردن بی‌صدا پنهان نشود. برای شمارنده‌ها صحیح‌بودن و نامنفی‌بودن بررسی می‌شود. صفر هزینهٔ حمل و پرداخت مجاز است؛ صفر تعداد اقساط یا وزن برای بارگذاری آینده مقدار نامعلوم همراه پرچم کیفیت پیشنهاد شده است.</p>

</div>


In [16]:
with pd.option_context("display.max_rows", None):
    display(inspection["schema"])
    display(inspection["numeric"])


,table,source,target,sql_type,nullable
0,customers,customer_id,customer_id,TEXT,False
1,customers,customer_unique_id,customer_unique_id,TEXT,False
2,customers,customer_zip_code_prefix,customer_zip_code_prefix,CHAR(5),False
3,customers,customer_city,customer_city,TEXT,False
4,customers,customer_state,customer_state,CHAR(2),False
5,orders,order_id,order_id,TEXT,False
6,orders,customer_id,customer_id,TEXT,False
7,orders,order_status,order_status,TEXT,False
8,orders,order_purchase_timestamp,order_purchase_timestamp,TIMESTAMP WITHOUT TIME ZONE,False
9,orders,order_approved_at,order_approved_at,TIMESTAMP WITHOUT TIME ZONE,True


,table,column,invalid_or_nonfinite_rows,min,max,negative_rows,zero_rows,fractional_integer_rows,below_rule_min_rows,above_rule_max_rows,integer_storage_overflow_rows,zero_to_null_rows,money_precision_errors
0,order_items,order_item_id,0,1.000000,21.000000,0,0,0,0,0,0,0,0
1,order_items,price,0,0.850000,6735.000000,0,0,0,0,0,0,0,0
2,order_items,freight_value,0,0.000000,409.680000,0,383,0,0,0,0,0,0
3,order_payments,payment_sequential,0,1.000000,29.000000,0,0,0,0,0,0,0,0
4,order_payments,payment_installments,0,0.000000,24.000000,0,2,0,0,0,0,2,0
5,order_payments,payment_value,0,0.000000,13664.080000,0,9,0,0,0,0,0,0
6,order_reviews,review_score,0,1.000000,5.000000,0,0,0,0,0,0,0,0
7,products,product_name_lenght,0,5.000000,76.000000,0,0,0,0,0,0,0,0
8,products,product_description_lenght,0,4.000000,3992.000000,0,0,0,0,0,0,0,0
9,products,product_photos_qty,0,1.000000,20.000000,0,0,0,0,0,0,0,0


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="schema-chronology">۲.۵. اعتبار تاریخ و ترتیب رویدادها</h3>

<p>تاریخ‌های منبع با قالب مشخص تبدیل آزمایشی می‌شوند و متن اصلی DataFrame حفظ می‌شود. تاریخ نامعتبر از رویداد خالی تفکیک شده است. برای پیشنهاد DATE، تعداد ساعت‌های غیرنیمه‌شب موعد تخمینی هم کنترل می‌شود.</p>

<p>ترتیب تأیید، ارسال و تحویل بدون بررسی به CHECK سراسری تبدیل نمی‌شود. زمان ناسازگار حفظ و علامت‌گذاری خواهد شد؛ مدت منفی در تحلیل همان مدت استفاده نمی‌شود. سفارش تحویل‌شده با تاریخ تحویل خالی، تأخیر صفر محسوب نمی‌شود.</p>

</div>


In [17]:
display(inspection["dates"])
display(inspection["chronology"])
display(pd.Series(inspection["special_counts"], name="observed_rows"))


,table,column,missing_rows,invalid_nonblank_rows,min,max,non_midnight_rows
0,orders,order_purchase_timestamp,0,0,2016-09-04T21:15:19,2018-10-17T17:30:18,NaN
1,orders,order_approved_at,160,0,2016-09-15T12:16:38,2018-09-03T17:40:06,NaN
2,orders,order_delivered_carrier_date,1783,0,2016-10-08T10:34:01,2018-09-11T19:48:28,NaN
3,orders,order_delivered_customer_date,2965,0,2016-10-11T13:46:32,2018-10-17T13:22:46,NaN
4,orders,order_estimated_delivery_date,0,0,2016-09-30T00:00:00,2018-11-12T00:00:00,0.0
5,order_items,shipping_limit_date,0,0,2016-09-19T00:15:34,2020-04-09T22:35:08,NaN
6,order_reviews,review_creation_date,0,0,2016-10-02T00:00:00,2018-08-31T00:00:00,NaN
7,order_reviews,review_answer_timestamp,0,0,2016-10-07T18:32:28,2018-10-29T12:27:35,NaN


,start,end,comparable_rows,negative_duration_rows
0,order_purchase_timestamp,order_approved_at,99281,0
1,order_approved_at,order_delivered_carrier_date,97644,1359
2,order_delivered_carrier_date,order_delivered_customer_date,96475,23
3,order_purchase_timestamp,order_delivered_customer_date,96476,0


delivered_without_actual_date    8
orders_without_payment           1
Name: observed_rows, dtype: int64

<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="schema-postal">۲.۶. قالب کد پستی و پوشش جغرافیایی</h3>

<p>کد پستی یک شناسه است و به‌صورت متن نگهداری می‌شود. پیش‌شمارهٔ یک تا پنج رقمی برای مقصد پنج‌رقمی از چپ با صفر کامل خواهد شد. اینجا فقط یک Series موقت برای بررسی ساخته می‌شود و فایل خام تغییر نمی‌کند.</p>

<p>وجود نداشتن یک پیش‌شماره در geolocation دلیل حذف مشتری یا فروشنده نیست. مرجع postal_areas از اجتماع پیش‌شماره‌ها پیشنهاد می‌شود و نقاط متعدد هر پیش‌شماره در جدول مستقل می‌مانند؛ پیش‌شماره کلید اصلی نقطه نیست.</p>

</div>


In [18]:
display(inspection["postal"])
print("Proposed postal_areas rows:", inspection["proposed_postal_area_rows"])


,table,missing_rows,invalid_nonblank_rows,needs_zero_padding_rows,missing_geolocation_rows
0,customers,0,0,0,278
1,sellers,0,0,0,7
2,geolocation,0,0,0,0


Proposed postal_areas rows: 19177


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="schema-evidence">۲.۷. ثبت شواهد و مرز مرحلهٔ فعلی</h3>

<p>این بخش بررسی داده و طرح منطقی را مستند می‌کند؛ هیچ جدول PostgreSQL یا دیتاست قرنطینه ساخته نشده است. هش ورودی‌ها دوباره کنترل می‌شود و جدول‌های نتایج در فایل شواهد ذخیره می‌شوند.</p>

<p>گزارش تصمیم‌ها به همین عنوان‌ها ارجاع دارد. بررسی خروجی NYC 311 در بخش‌های ۲.۱۱ تا ۲.۱۵ آمده است؛ پیاده‌سازی DDL و بارگذاری گام بعد هستند. شواهد قدیمیِ بررسی مستقل، مبنای ادعای اجرای این نسخه نیستند.</p>

</div>


In [19]:
evidence_path = save_evidence(PROJECT_ROOT, inspection)
print("Saved:", evidence_path.relative_to(PROJECT_ROOT).as_posix())
print("Raw CSV hashes unchanged; no database loading or quarantine executed.")


Saved: stages/stage_02/evidence/quality/notebook_checks.json
Raw CSV hashes unchanged; no database loading or quarantine executed.


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="cleaning-run">۲.۸. پاک‌سازی و اعتبارسنجی پیش از بارگذاری</h3>

<p>این سلول واقعاً مرحلهٔ پردازش را اجرا می‌کند. ورودی CSV در همان پکیج کارخانه خوانده می‌شود و منطق پاک‌سازی در <code dir="ltr">data_factory.olist_cleaning</code> قرار دارد. فایل‌های raw تغییر نمی‌کنند؛ خروجی‌ها در مسیر curated و پوشهٔ وابسته به هش ورودی، قرارداد و کد منتشر می‌شوند.</p>

<p>تکرار کامل geolocation بر اساس همهٔ ستون‌های نقطه حذف می‌شود؛ نقطه‌های متفاوت با پیش‌شمارهٔ یکسان باقی می‌مانند. مقدارهای نامعلوم و متنِ صرفاً خالی به NULL تبدیل می‌شوند؛ کلید یا مقدار ضروری نامعتبر پیش از خروجی قرنطینه می‌شود و خطای والد به فرزندان وابسته منتقل می‌شود.</p>

<p>جدول شمارش، ردیف خام، ردیف پذیرفته‌شدهٔ منبع، حذف تکرار، قرنطینه، ردیف تولیدشده و خروجی نهایی را جدا نشان می‌دهد. جدول دسته‌ها با مقادیر بدون ترجمه تکمیل می‌شود و جدول postal_areas نیز تولید می‌شود؛ بنابراین افزایش این دو خروجی عمدی و قابل ردیابی است.</p>

</div>


In [20]:
from data_factory.olist_cleaning import run_olist_cleaning

cleaning = run_olist_cleaning(PROJECT_ROOT)
cleaning_report = cleaning["report"]
display(pd.DataFrame(cleaning_report["table_rows"])[[
    "table", "raw_rows", "accepted_source_rows", "deduplicated_rows",
    "quarantined_rows", "generated_rows", "output_rows",
]])
print("Curated output:", Path(cleaning["output_dir"]).relative_to(PROJECT_ROOT).as_posix())

,table,raw_rows,accepted_source_rows,deduplicated_rows,quarantined_rows,generated_rows,output_rows
0,customers,99441,99441,0,0,0,99441
1,orders,99441,99441,0,0,0,99441
2,order_items,112650,112650,0,0,0,112650
3,order_payments,103886,103886,0,0,0,103886
4,order_reviews,99224,99224,0,0,0,99224
5,products,32951,32951,0,0,0,32951
6,sellers,3095,3095,0,0,0,3095
7,geolocation,1000163,738332,261831,0,0,738332
8,category_translation,71,71,0,0,2,73
9,postal_areas,0,0,0,0,19177,19177


Curated output: data/stage_02/curated/olist/0d73c90d287644bd4c0e93bb1c93882c7abf80479340c08e247bc36bc943eb7d


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="cleaning-changes">۲.۹. تغییر مقدارها و پرچم‌های کیفیت</h3>

<p>جدول اول تغییرهای مشخص را نشان می‌دهد: صفرِ اقساط و وزن به مقدار نامعلوم تبدیل شده، متنِ فقط فاصله NULL شده و فاصلهٔ اضافی متن‌های کلیدی اصلاح شده است. متن واقعی نظرها، مبلغ صفر و هزینهٔ حمل صفر حفظ می‌شوند؛ مقدارهای اختیاری خالی با حدس پر نمی‌شوند.</p>

<p>پرچم زمانی باعث حذف کل سفارش نمی‌شود. ستون‌های اعتبار هر مدت و <code dir="ltr">delivery_delay_eligible</code> در خروجی ثبت شده‌اند؛ معتبر بودن یک معیار مستقل از معیار دیگر است. شمار پرچم‌ها به تفکیک جدول نمایش داده می‌شود و نباید برای محاسبهٔ تعداد رکوردهای یکتا با هم جمع شوند.</p>

</div>


In [21]:
cleaning_changes = pd.DataFrame(cleaning_report["column_changes"])
changed_rows = cleaning_changes[[
    "whitespace_only_to_null_rows", "trimmed_nonblank_rows", "zero_to_null_rows",
    "invalid_optional_to_null_rows", "zero_padded_rows",
]].sum(axis=1).gt(0)
display(cleaning_changes.loc[changed_rows])
flag_table = pd.DataFrame(cleaning_report["quality_flags_by_table"]).T.fillna(0).astype(int)
display(flag_table.rename(index={"category_translation": "product_categories"}))

,table,column,target,blank_to_null_rows,whitespace_only_to_null_rows,trimmed_nonblank_rows,zero_to_null_rows,invalid_optional_to_null_rows,zero_padded_rows,invalid_required_rows
23,order_payments,payment_installments,payment_installments,0,0,0,2,0,0,0
28,order_reviews,review_comment_title,review_comment_title,87658,2,0,0,0,0,0
29,order_reviews,review_comment_message,review_comment_message,58274,27,0,0,0,0,0
37,products,product_weight_g,product_weight_g,2,0,0,4,0,0,0
48,geolocation,geolocation_city,geolocation_city,0,0,1,0,0,0,0


,NO_GEOLOCATION_MATCH,CARRIER_BEFORE_APPROVAL,CUSTOMER_BEFORE_CARRIER,DELIVERED_WITHOUT_ACTUAL_DATE,ORDER_WITHOUT_PAYMENT,UNKNOWN_ZERO:payment_installments,MISSING_CATEGORY_TRANSLATION,UNKNOWN_ZERO:product_weight_g
customers,278,0,0,0,0,0,0,0
orders,0,1359,23,8,1,0,0,0
order_items,0,0,0,0,0,0,0,0
order_payments,0,0,0,0,0,2,0,0
order_reviews,0,0,0,0,0,0,0,0
products,0,0,0,0,0,0,13,4
sellers,7,0,0,0,0,0,0,0
geolocation,0,0,0,0,0,0,0,0
product_categories,0,0,0,0,0,0,2,0


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3 id="cleaning-validation">۲.۱۰. کنترل خروجی، منشأ و تکرارپذیری</h3>

<p>پس از پردازش، کلیدهای اصلی، مقدارهای الزامی و ارجاع‌ها دوباره کنترل شده‌اند. شمارهٔ source_row، شمارهٔ رکورد دادهٔ CSV بدون هدر است؛ برای CSV دارای متن چندخطی، با شمارهٔ خط فایل یکی نیست. quality_flags یک آرایهٔ JSON است و نگاشت نوع همهٔ ستون‌های خروجی در گزارش ماشینی ثبت می‌شود.</p>

<p>فایل deduplicated.jsonl رکورد حذف‌شده را به رکورد حفظ‌شده وصل می‌کند. quarantine.jsonl رکورد خام نامعتبر و دلیل‌ها را نگه می‌دارد؛ خالی بودن آن نیز نتیجهٔ واقعی اجراست. generated.jsonl منشأ دسته‌های افزوده را ثبت می‌کند. هش خروجی‌ها و زمان واقعی هر اجرا در manifest قرار دارند.</p>

<p>اجرای دوبارهٔ ورودی و قرارداد و کد یکسان، همان فایل‌های داده را تولید می‌کند؛ شناسه و زمان manifest اجرایی عمداً متفاوت‌اند. این مرحله هنوز هیچ داده‌ای در PostgreSQL درج نمی‌کند. بررسی منبع دوم در بخش بعد آمده است؛ نوشتن DDL و بارگذاری باقی مانده‌اند.</p>

</div>


In [22]:
display(pd.DataFrame(cleaning_report["post_checks"]["keys"]))
display(pd.Series({
    "status": cleaning_report["post_checks"]["status"],
    "quarantined_rows": cleaning_report["quarantined_rows"],
    "unmatched_foreign_key_rows": cleaning_report["post_checks"]["unmatched_foreign_key_rows"],
    "remaining_exact_geolocation_duplicates": cleaning_report["post_checks"]["geolocation_exact_duplicate_rows"],
    "database_writes": cleaning_report["database_writes"],
}, name="cleaning_output_checks"))
print("Cleaning report:", Path(cleaning["evidence_path"]).relative_to(PROJECT_ROOT).as_posix())
print("Execution manifest:", Path(cleaning["manifest_path"]).relative_to(PROJECT_ROOT).as_posix())

,table,missing_keys,duplicate_keys,missing_required_values
0,customers,0,0,0
1,orders,0,0,0
2,order_items,0,0,0
3,order_payments,0,0,0
4,order_reviews,0,0,0
5,products,0,0,0
6,sellers,0,0,0
7,geolocation,0,0,0
8,category_translation,0,0,0


status                                    passed
quarantined_rows                               0
unmatched_foreign_key_rows                     0
remaining_exact_geolocation_duplicates         0
database_writes                                0
Name: cleaning_output_checks, dtype: object

Cleaning report: stages/stage_02/evidence/quality/cleaning_report.json
Execution manifest: data/stage_02/curated/olist/executions/02a232e67bf342dd802ad9da1d62e376/run_manifest.json


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">
<h3 id="nyc-preview">۲.۱۱. خواندن خروجی NYC 311 و اصلاح مسیر</h3>
<p>فایل در <code dir="ltr">data/curated_data/delivery/&lt;run-hash&gt;/analytical_cases.jsonl</code> است؛ مسیر قبلی پوشهٔ delivery را نداشت. مسیر نسبی با محل اجرای Python تفسیر می‌شود؛ اینجا ریشهٔ پروژه را سلول آماده‌سازی پیدا کرده است. شناسهٔ اجرا را از latest.json می‌خوانیم و هش فایل و گزارش را با manifest تطبیق می‌دهیم تا مسیر وابسته به یک هش ثابت نباشد.</p>
<p>در JSONL هر خط یک رکورد JSON است؛ پس <code dir="ltr">lines=True</code> لازم است. فایل حدود ۳۸۰ مگابایت است. متغیر <code dir="ltr">NYC_data</code> فقط اولین بستهٔ حداکثر ۵۰۰۰ ردیفی برای مشاهده است؛ بررسی کامل در سلول بعد همهٔ بسته‌ها را می‌خواند. شناسه و ZIP رشته باقی می‌مانند و تاریخ‌ها برای بررسی صریح، خودکار تبدیل نمی‌شوند.</p>
</div>


In [23]:
from data_factory.database.nyc_inspection import (
    resolve_nyc_delivery, read_nyc_batches, inspect_nyc_delivery, save_nyc_evidence,
)

nyc_source = resolve_nyc_delivery(PROJECT_ROOT)
print("Resolved JSONL:", nyc_source["path"].relative_to(PROJECT_ROOT).as_posix())
with read_nyc_batches(nyc_source, chunksize=5000) as batches:
    NYC_data = next(batches)
print("Preview only:", NYC_data.shape)
display(NYC_data.head())


Resolved JSONL: data/curated_data/delivery/02fc160e6673cfd7f52b2ded9cb48057eec9f46af139add0bd0cba015bdbccef/analytical_cases.jsonl
Preview only: (5000, 32)


,agency,agency_name,analysis_area_key,borough,borough_comparison,city,closed_date,closure_days,closure_hours,complaint_type,...,population_rate_eligible,population_reference_date,quality_flags,reference_borough,reference_neighborhood,reference_population,resolution_action_updated_date,resolution_description,status,unique_key
0,NYPD,New York City Police Department,MODZCTA:10468,BRONX,match,BRONX,2024-01-01T00:59:31.000000,0.041100,0.986389,Noise - Residential,...,True,2019-07-01,[],BRONX,Fordham/Kingsbridge/University Heights,77811.41,2024-01-01T00:59:34.000000,The Police Department responded to the complai...,Closed,59889016
1,NYPD,New York City Police Department,MODZCTA:11207,BROOKLYN,match,BROOKLYN,2024-01-01T00:56:45.000000,0.038912,0.933889,Noise - Residential,...,True,2019-07-01,[],BROOKLYN,Cypress Hills/East New York,89031.83,2024-01-01T00:56:50.000000,The Police Department responded to the complai...,Closed,59888940
2,NYPD,New York City Police Department,MODZCTA:11377,QUEENS,match,WOODSIDE,2024-01-01T00:48:34.000000,0.033171,0.796111,Noise - Street/Sidewalk,...,True,2019-07-01,[],QUEENS,Woodside,84048.11,2024-01-01T00:48:39.000000,The Police Department responded to the complai...,Closed,59892651
3,NYPD,New York City Police Department,MODZCTA:11416,QUEENS,match,OZONE PARK,2024-01-01T00:29:32.000000,0.019919,0.478056,Noise - Street/Sidewalk,...,True,2019-07-01,[],QUEENS,Ozone Park,26389.62,2024-01-01T00:29:34.000000,The Police Department responded to the complai...,Closed,59889054
4,NYPD,New York City Police Department,MODZCTA:11434,QUEENS,match,JAMAICA,2024-01-01T01:32:22.000000,0.063437,1.522500,Noise - Street/Sidewalk,...,True,2019-07-01,[],QUEENS,Airport/South Jamaica/Springfield Gardens/St. ...,66763.37,2024-01-01T01:32:26.000000,The Police Department responded to the complai...,Closed,59892073


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">
<h3 id="nyc-checks">۲.۱۲. بررسی کامل ساختار، کلیدها و مقدارهای مشتق</h3>
<p>منطق در اسکریپت <code dir="ltr">data_factory.database.nyc_inspection</code> قرار دارد و اینجا فقط فراخوانی می‌شود. تمام رکوردهای ذخیره‌شده، در بسته‌های ۱۰۰۰۰تایی، از نظر کلید unique_key، مقدار الزامی، قالب تاریخ و ZIP، مختصات و فهرست پرچم‌ها بررسی می‌شوند. یکتایی کلید بین بسته‌ها هم کنترل می‌شود؛ بنابراین حافظهٔ نگهداری کلیدها با تعداد کلیدهای یکتا رشد می‌کند.</p>
<p>ستون‌های closure_hours، closure_days، created_month و has_recorded_closure با تاریخ‌ها تطبیق داده می‌شوند. خروجی stage 01 قبلاً پاک‌سازی شده است؛ این بررسی داده‌ای را حذف یا جایگزین نمی‌کند. صفر بودن خطا به معنی کامل بودن دریافت نیست؛ دامنه و وضعیت delivery_complete جدا نمایش داده می‌شوند.</p>
</div>


In [24]:
nyc_inspection = inspect_nyc_delivery(PROJECT_ROOT, source=nyc_source, chunksize=10000)
display(pd.Series({key: nyc_inspection[key] for key in (
    "status", "rows", "source_columns", "missing_primary_keys",
    "duplicate_primary_key_extra_rows", "missing_required_values", "delivery_complete",
)}, name="full_saved_artifact"))
display(pd.DataFrame(nyc_inspection["dates"]).T)
print(nyc_inspection["scope_note"])
display(pd.Series(nyc_inspection["consistency_checks"], name="violations"))
assert nyc_inspection["status"] == "passed", "Review NYC diagnostics before database loading."


status                              passed
rows                                330000
source_columns                          32
missing_primary_keys                     0
duplicate_primary_key_extra_rows         0
missing_required_values                  0
delivery_complete                    False
Name: full_saved_artifact, dtype: object

,min,max
closed_date,2024-01-01T00:12:42,2024-07-26T22:13:46
created_date,2024-01-01T00:00:20,2024-07-25T23:32:51
population_reference_date,2019-07-01T00:00:00,2019-07-01T00:00:00
resolution_action_updated_date,2024-01-01T00:12:46,2024-07-26T22:13:49


Partial acquisition; the full selected 18-month scope is not yet available.


unknown_population_area_rows                  0
population_value_mismatch_rows                0
reference_attribute_mismatch_rows             0
population_alias_mismatch_rows                0
population_join_status_mismatch_rows          0
rate_eligibility_mismatch_rows                0
negative_closure_rows                         0
closure_hours_mismatch_rows                   0
closure_days_mismatch_rows                    0
closure_flag_mismatch_rows                    0
created_month_mismatch_rows                   0
reference_matched_case_count_mismatch_rows    0
Name: violations, dtype: int64

<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">
<h3 id="nyc-types">۲.۱۳. نوع ستون‌ها و پیشنهاد منطقی schema</h3>
<p>جدول زیر نوع منطقی و SQL پیشنهادی، nullability، جدول مقصد و تعداد مقدار خالی/نامعتبر هر ستون را نشان می‌دهد. کلید unique_key متن است؛ کیفیت‌ها JSONB و تاریخ‌ها بدون افزودن منطقهٔ زمانی فرضی نگهداری می‌شوند. جمعیت NUMERIC(14,2) است، زیرا مرجع جمعیت مقدار اعشاری دارد. این فایل قرارداد پیشنهادی است و هنوز DDL اجرا نشده است.</p>
<p>پیشنهاد در <code dir="ltr">config/nyc_schema.json</code> سه جدول دارد: cases با ۲۷ ستون، population_areas با ۶ ستون و population_zip_aliases با ۲ ستون. پنج ویژگی مرجع تکرارشوندهٔ هر درخواست در جدول جمعیت ذخیره می‌شوند و از population_area قابل اتصال‌اند. matched_case_count یک شاخص snapshot است و ویژگی ثابت جمعیت محسوب نمی‌شود. جدول alias اجازه می‌دهد ZIP با MODZCTA یکسان فرض نشود.</p>
</div>


In [25]:
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame(nyc_inspection["fields"]))
display(pd.DataFrame([
    {"table": name, "columns": len(spec.get("source_columns", spec.get("columns", {}))),
     "primary_key": ", ".join(spec["primary_key"]), "foreign_keys": spec.get("foreign_keys", [])}
    for name, spec in nyc_source["proposal"]["tables"].items()
]))


,source,kind,sql_type,nullable,target_table,primary_key,missing_rows,invalid_rows,maximum_text_length
0,agency,string,TEXT,False,cases,False,0,0,5
1,agency_name,string,TEXT,True,cases,False,0,0,31
2,analysis_area_key,string,TEXT,False,cases,False,0,0,19
3,borough,string,TEXT,True,cases,False,13,0,13
4,borough_comparison,string,TEXT,False,cases,False,0,0,11
5,city,string,TEXT,True,cases,False,14418,0,19
6,closed_date,datetime,TIMESTAMP WITHOUT TIME ZONE,True,cases,False,1,0,26
7,closure_days,float,DOUBLE PRECISION,True,cases,False,1,0,0
8,closure_hours,float,DOUBLE PRECISION,True,cases,False,1,0,0
9,complaint_type,string,TEXT,False,cases,False,0,0,23


,table,columns,primary_key,foreign_keys
0,cases,27,unique_key,"[{'column': 'population_area', 'parent': 'popu..."
1,population_areas,6,population_area,[]
2,population_zip_aliases,2,incident_zip,"[{'column': 'population_area', 'parent': 'popu..."


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">
<h3 id="nyc-reference">۲.۱۴. مرجع جمعیت و رکوردهای فاقد اتصال</h3>
<p>مرجع ذخیره‌شدهٔ همان اجرای مرحلهٔ اول با هش گزارش اتصال انتخاب می‌شود: ۱۷۷ ناحیهٔ جمعیت و ۱۸۳ ZIP alias. یکتایی کلیدهای مرجع، قالب کدها، مقدار جمعیت و سازگاری ویژگی‌های مرجع با هر درخواست کنترل می‌شود. ناموفق بودن اتصال باعث حذف درخواست نمی‌شود؛ population_area می‌تواند NULL باشد.</p>
<p>در snapshot فعلی ۳۲۹۶۷۲ درخواست متصل، ۶۷ مورد بدون ZIP و ۲۶۱ مورد با ZIP ناموجود در مرجع داریم. ۷۷ ناسازگاری borough حفظ شده‌اند و برای نرخ مبتنی بر جمعیت مجاز نیستند. یک درخواست تاریخ بسته‌شدن ندارد؛ مدت آن NULL می‌ماند. تاریخ مرجع جمعیت ۲۰۱۹ است و نباید جمعیت روزِ رویداد در ۲۰۲۴ تلقی شود.</p>
</div>


In [26]:
display(pd.Series({"population_areas": nyc_inspection["reference_rows"],
                   "zip_aliases": nyc_inspection["alias_rows"]}, name="reference_rows"))
display(pd.Series(nyc_inspection["join_status_counts"], name="population_join"))
display(pd.Series(nyc_inspection["borough_comparison_counts"], name="borough_comparison"))
display(pd.Series(nyc_inspection["reference_checks"], name="reference_violations"))


population_areas    177
zip_aliases         183
Name: reference_rows, dtype: int64

matched          329672
missing_zip          67
unmatched_zip       261
Name: population_join, dtype: int64

match          329584
mismatch           77
unavailable       339
Name: borough_comparison, dtype: int64

missing_area_keys                    0
duplicate_area_keys                  0
invalid_area_keys                    0
duplicate_zip_aliases                0
invalid_zip_aliases                  0
invalid_reference_population         0
missing_required_reference_values    0
Name: reference_violations, dtype: int64

<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">
<h3 id="nyc-evidence">۲.۱۵. ثبت شواهد منبع دوم و گام بعد</h3>
<p>هش فایل‌های منبع، شمار همهٔ رکوردها، نوع‌ها، کلیدها، تطبیق مرجع و وضعیت ناقص بودن دریافت در evidence ذخیره می‌شوند. برای این بررسی هیچ درخواست شبکه‌ای یا نوشتن دیتابیس انجام نشده است. پوشش رویدادهای این فایل از ۱ ژانویه تا ۲۵ ژوئیهٔ ۲۰۲۴ است؛ پوشش کامل ۱۸ ماه ادعا نمی‌شود. گام بعد تبدیل قرارداد دو منبع به DDL و سپس پیاده‌سازی بارگذاری PostgreSQL در کارخانه است.</p>
</div>


In [27]:
nyc_evidence_path = save_nyc_evidence(PROJECT_ROOT, nyc_inspection)
print("Saved:", nyc_evidence_path.relative_to(PROJECT_ROOT).as_posix())
print("Database writes:", nyc_inspection["database_writes"])


Saved: stages/stage_02/evidence/quality/nyc_source_checks.json
Database writes: 0


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۳. ساخت ساختار و بارگذاری داده‌ها</h2>

<p>ورودی <bdi dir="ltr">CSV</bdi> در <code dir="ltr" style="unicode-bidi: isolate; display: inline-block;">src/data_factory/sources/</code> و مقصد دیتابیس در <code dir="ltr" style="unicode-bidi: isolate; display: inline-block;">src/data_factory/sinks/</code> توسعه خواهند یافت. اعتبارسنجی، قرنطینه و <bdi dir="ltr">manifest</bdi> باید از منطق مشترک کارخانه استفاده کنند. شمار رکوردهای خام، پذیرفته‌شده، ردشده و بارگذاری‌شده ثبت خواهد شد.</p>

<p>وضعیت: اجرای DDL در بخش ۳.۱ تکمیل شده است؛ بارگذاری داده‌ها در بخش ۳.۲ باقی مانده است.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3>۳.۱. اجرای DDL و ساخت جدول‌ها</h3>
<p>Engine در بخش ۱.۱ ساخته شده است. اینجا دو فایل <code dir="ltr">sql/ddl/001_olist.sql</code> و <code dir="ltr">sql/ddl/002_nyc311.sql</code> به‌ترتیب، از طریق همان Engine و در یک تراکنش اجرا می‌شوند. اگر هر بخش خطا دهد، تمام تغییرات این اجرا به عقب برمی‌گردند.</p>
<p>ماژول <code dir="ltr">data_factory.database.ddl</code> پیش از اجرا وجود جدول‌ها را بررسی می‌کند. اگر هیچ‌کدام از ۱۲ جدول مورد انتظار نباشند، ساخت انجام می‌شود. اگر همهٔ جدول‌ها موجود باشند، وضعیت <code dir="ltr">already_present</code> گزارش می‌شود و هیچ فایل DDL دوباره اجرا نمی‌شود. ساختار ناقص یا برخورد نام جدول با یک شیء دیگر، باعث توقف پیش از اجرای فایل‌ها می‌شود.</p>
<p>این مسیر برای ساخت اولیه است: در اجرای مجدد، تعریف موجود جدول‌ها با نسخهٔ تغییریافتهٔ SQL مقایسه یا به‌روز نمی‌شود. کلیدها و وضعیت فعلی جدول‌ها در خروجی بررسی می‌شوند. هیچ جدول موجودی حذف یا بازسازی نمی‌شود و داده‌ای بارگذاری نمی‌شود.</p>

</div>


In [28]:
import hashlib
import json
import pandas as pd
from data_factory.database.ddl import run_initial_ddl, inspect_initial_schema

ddl_report = run_initial_ddl(engine, PROJECT_ROOT / "sql/ddl")
ddl_report["implementation"] = {
    "file": "src/data_factory/database/ddl.py",
    "sha256": hashlib.sha256((PROJECT_ROOT / "src/data_factory/database/ddl.py").read_bytes()).hexdigest(),
}
ddl_evidence_path = PROJECT_ROOT / "stages/stage_02/evidence/quality/ddl_execution.json"
ddl_evidence_path.parent.mkdir(parents=True, exist_ok=True)
ddl_evidence_path.write_text(
    json.dumps(ddl_report, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print("DDL status:", ddl_report["status"])
print("Database:", ddl_report["identity"]["database_name"])
print("Transaction committed:", ddl_report["transaction_committed"])
print("Evidence:", ddl_evidence_path.relative_to(PROJECT_ROOT).as_posix())
display(pd.DataFrame(ddl_report["files"])[["file", "executed"]])


DDL status: already_present
Database: ds_learning
Transaction committed: True
Evidence: stages/stage_02/evidence/quality/ddl_execution.json


,file,executed
0,001_olist.sql,False
1,002_nyc311.sql,False


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h4>بررسی ساختار ثبت‌شده پس از commit</h4>
<p>سلول بعد در یک اتصال تازه، وجود جدول‌ها، ستون‌ها و محدودیت‌های واقعی PostgreSQL را می‌خواند. شمار رکورد هر جدول نیز دقیقاً محاسبه می‌شود. صفر بودن این شمار در اجرای اولیه یعنی ساختار ساخته شده و مرحلهٔ بارگذاری هنوز آغاز نشده است. پس از بارگذاری‌های آینده، اجرای مجدد همین سلول تعداد واقعی داده‌های موجود را نشان می‌دهد.</p>

</div>


In [29]:
committed_schema = inspect_initial_schema(engine)
ddl_report["post_commit_verified"] = True
ddl_report["post_commit_table_count"] = committed_schema["table_count"]
ddl_evidence_path.write_text(
    json.dumps(ddl_report, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print("Committed tables:", committed_schema["table_count"])
print("Total data rows:", sum(row["row_count"] for row in committed_schema["tables"]))
table_summary = pd.DataFrame(committed_schema["tables"])
table_summary["primary_key"] = table_summary["primary_key"].map(", ".join)
display(table_summary)
display(pd.DataFrame.from_dict(committed_schema["constraint_counts"], orient="index", columns=["count"]).rename_axis("constraint_type"))


Committed tables: 12
Total data rows: 0


,schema,table,column_count,primary_key,foreign_key_count,unique_count,check_count,row_count
0,olist,product_categories,4,product_category_name,0,0,3,0
1,olist,customers,7,customer_id,0,0,7,0
2,olist,sellers,6,seller_id,0,0,6,0
3,olist,products,11,product_id,1,0,10,0
4,olist,geolocation,8,geo_point_id,0,1,7,0
5,olist,orders,15,order_id,1,0,4,0
6,olist,order_items,9,"order_id, order_item_id",3,0,5,0
7,olist,order_payments,7,"order_id, payment_sequential",1,0,6,0
8,olist,order_reviews,9,"review_id, order_id",1,0,4,0
9,nyc311,population_areas,6,population_area,0,0,4,0


,count
constraint_type,
PRIMARY KEY,12
CHECK,75
FOREIGN KEY,9
UNIQUE,1


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<p>جزئیات ستون‌ها، نوع داده، nullable، مقدار پیش‌فرض، identity و محدودیت‌های PRIMARY KEY، FOREIGN KEY، UNIQUE و CHECK در فایل شواهد ثبت می‌شوند. وضعیت NOT NULL در ویژگی nullable ستون‌ها گزارش می‌شود. این بررسی شامل بارگذاری داده، ساخت View یا سنجش کارایی نیست.</p>
<p>سلول آماده‌سازی پروژه، اتصال واقعی، دو سلول DDL/بررسی و آزادسازی pool در یک فرایند تازهٔ IPython اجرا شدند. سایر سلول‌ها و خروجی‌های بررسی و پاک‌سازی داده‌ها حفظ شده‌اند؛ اجرای کامل نوت‌بوک یا Jupyter kernel انجام نشده است.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3>۳.۲. بارگذاری داده‌ها — گام بعدی</h3>
<p>ساختار دیتابیس آماده است. بارگذاری خروجی‌های آماده‌شدهٔ Olist و NYC 311، ترتیب درج با توجه به کلیدهای خارجی، مدیریت تکرارها و گزارش تعداد رکوردها در گام بعدی پیاده‌سازی می‌شوند.</p>

</div>


In [30]:
import csv
import json
from psycopg import sql

base = PROJECT_ROOT / "data/stage_02/curated/olist"
latest = json.loads((base / "latest.json").read_text(encoding="utf-8"))
data_dir = base / latest["run"]

tables = [
    "product_categories", "customers", "sellers", "products",
    "geolocation", "orders", "order_items",
    "order_payments", "order_reviews",
]

loaded = {}

with engine.begin() as connection:
    with connection.connection.driver_connection.cursor() as cursor:
        for table_name in tables:
            path = data_dir / f"{table_name}.csv"

            with path.open("r", encoding="utf-8", newline="") as file:
                columns = next(csv.reader(file))
                file.seek(0)

                command = sql.SQL(
                    "COPY {} ({}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE)"
                ).format(
                    sql.Identifier("olist", table_name),
                    sql.SQL(", ").join(sql.Identifier(c) for c in columns),
                )

                with cursor.copy(command) as copy:
                    while chunk := file.read(1024 * 1024):
                        copy.write(chunk)

                loaded[table_name] = cursor.rowcount

for table_name, count in loaded.items():
    print(f"{table_name}: {count:,} rows loaded")

product_categories: 73 rows loaded
customers: 99,441 rows loaded
sellers: 3,095 rows loaded
products: 32,951 rows loaded
geolocation: 738,332 rows loaded
orders: 99,441 rows loaded
order_items: 112,650 rows loaded
order_payments: 103,886 rows loaded
order_reviews: 99,224 rows loaded


In [ ]:
base = PROJECT_ROOT / "data/stage_02/curated/olist"
latest = json.loads((base / "latest.json").read_text(encoding="utf-8"))
data_dir = base / latest["run"]

tables = [
    "product_categories", "customers", "sellers", "products",
    "geolocation", "orders", "order_items",
    "order_payments", "order_reviews",
]

loaded = {}

with engine.begin() as connection:
    with connection.connection.driver_connection.cursor() as cursor:
        for table_name in tables:
            path = data_dir / f"{table_name}.csv"

            with path.open("r", encoding="utf-8", newline="") as file:
                columns = next(csv.reader(file))
                file.seek(0)

                command = sql.SQL(
                    "COPY {} ({}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE)"
                ).format(
                    sql.Identifier("olist", table_name),
                    sql.SQL(", ").join(sql.Identifier(c) for c in columns),
                )

                with cursor.copy(command) as copy:
                    while chunk := file.read(1024 * 1024):
                        copy.write(chunk)

                loaded[table_name] = cursor.rowcount

for table_name, count in loaded.items():
    print(f"{table_name}: {count:,} rows loaded")

<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۴. پنج سؤال تحلیلی و خط مبنای کارایی</h2>

<p>کوئری‌های سبد و درآمد، تأخیر تحویل، رضایت، فروشندگان برتر و روش پرداخت در <code dir="ltr" style="unicode-bidi: isolate; display: inline-block;">sql/queries/</code> نوشته خواهند شد. تعریف معیارها و رفتار مقادیر خالی باید روشن باشد. برای هر کوئری، نتیجه، روش اندازه‌گیری تکرارپذیر، زمان و پلن اولیه ثبت خواهد شد.</p>

<p>وضعیت: انجام نشده.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۵. بهینه‌سازی و هزینهٔ ایندکس</h2>

<p>برای هر تغییر، بخش پرهزینهٔ پلن، تخمین و تعداد واقعی سطرها و علت بهبود بررسی خواهد شد. هر ایندکس به شاهد نیاز دارد. حجم ایندکس‌ها و اثر آن‌ها بر درج دسته‌ای اندازه‌گیری خواهد شد و حداقل یک ایندکس کنارگذاشته‌شده مستند می‌شود. شواهد در <code dir="ltr" style="unicode-bidi: isolate; display: inline-block;">stages/stage_02/evidence/</code> قرار می‌گیرند.</p>

<p>وضعیت: انجام نشده.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۶. بهبود فراتر از ایندکس</h2>

<p>حداقل یک تحلیل باید با تغییری فراتر از افزودن ایندکس به بودجه برسد. در صورت نیاز، سؤال ششم طراحی خواهد شد. پلن باید نشان دهد چرا ایندکس کافی نبوده است و راه‌حل نهایی چه هزینه‌ای داشته است.</p>

<p>وضعیت: انجام نشده.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۷. <bdi dir="ltr">View</bdi> و مقایسه با <bdi dir="ltr">snapshot</bdi></h2>

<p>یک <bdi dir="ltr">View</bdi> ساده و یک <bdi dir="ltr">View</bdi> چندجدولی ساخته خواهند شد و حداقل یک سؤال از طریق <bdi dir="ltr">View</bdi> پاسخ داده می‌شود. برای سنگین‌ترین تحلیل، زمان <bdi dir="ltr">View</bdi> زنده و جدول <bdi dir="ltr">snapshot</bdi> مقایسه خواهد شد؛ تازگی داده، فضا و نگهداری در تصمیم نهایی لحاظ می‌شوند.</p>

<p>وضعیت: انجام نشده.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۸. <bdi dir="ltr">Function</bdi> و <bdi dir="ltr">Procedure</bdi></h2>

<p>یک <bdi dir="ltr">Function</bdi> قابل استفاده در <bdi dir="ltr">SELECT</bdi> و یک <bdi dir="ltr">Procedure</bdi> پارامتری برای بازسازی <bdi dir="ltr">snapshot</bdi> ماهانه نوشته خواهند شد. اجرای دوبارهٔ <bdi dir="ltr">Procedure</bdi> نباید دادهٔ تکراری بسازد و خطای میانی نباید خروجی ناقص باقی بگذارد. تفاوت این دو با ارجاع به کد واقعی توضیح داده خواهد شد.</p>

<p>وضعیت: انجام نشده.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۹. الگوهای تعامل Python و SQL</h2>
<p>ساخت Engine و بررسی اتصال واقعی در بخش ۱.۱، پیش از بخش منابع و طراحی schema انجام می‌شود. تمام اجرای SQL از متغیر مشترک <code dir="ltr">engine</code> و از طریق SQLAlchemy استفاده می‌کند؛ این بخش محل تعریف الگوهای خواندن، تراکنش، پارامترهای bind، خواندن دسته‌ای و نگاشت نوع ستون‌ها است.</p>
<p>فایل‌های <code dir="ltr">sql/</code> تعریف‌های نسخه‌بندی‌شدهٔ schema، جدول، محدودیت‌ها و کوئری‌ها را نگه می‌دارند. اجرای تغییرات با <code dir="ltr">engine.begin()</code> و خواندن داده‌ها با <code dir="ltr">engine.connect()</code> انجام می‌شود. Connection هر بلوک پس از پایان کار آزاد می‌شود؛ pool در سلول پایانی نوت‌بوک آزاد می‌شود.</p>
<p>وضعیت: تنظیمات، اتصال واقعی و اجرای DDL آماده‌اند؛ بارگذاری، کوئری‌های تحلیلی و الگوهای خواندن/نوشتن داده در گام‌های بعدی تکمیل می‌شوند.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۱۰. تست‌ها و بررسی پذیرش</h2>

<p>حداقل شش تست معنادار در <code dir="ltr" style="unicode-bidi: isolate; display: inline-block;">tests/database/</code> نوشته خواهد شد. درستی <bdi dir="ltr">Function</bdi>، اجرای دوبارهٔ <bdi dir="ltr">Procedure</bdi>، <bdi dir="ltr">rollback</bdi> و نتیجهٔ یک کوئری روی دادهٔ کوچک کنترل‌شده بررسی خواهند شد. سپس اجرای کامل از دیتابیس خالی و بودجهٔ زمانی کوئری‌ها ارزیابی می‌شوند.</p>

<p>وضعیت: انجام نشده.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h2>۱۱. تصمیم‌ها، محدودیت‌ها و وضعیت تحویل</h2>

<p>پس از اجرای تحلیل‌ها، نتیجه‌ها و موارد باقی‌مانده اینجا ثبت می‌شوند. گزارش تصمیم‌ها در <a href="report.md"><bdi dir="ltr">report</bdi>.<bdi dir="ltr">md</bdi></a> قرار دارد و همهٔ اعداد آن باید به خروجی‌های همین نوت‌بوک قابل ردیابی باشند. بررسی و پاک‌سازی منبع Olist انجام شده و خروجی curated آماده است؛ منبع دوم و بارگذاری و تحلیل SQL باقی مانده‌اند.</p>

<p>اتصال و اجرای DDL در دیتابیس واقعی انجام شده و شواهد ساختار در <code dir="ltr">evidence/quality/ddl_execution.json</code> ثبت شده است. جدول‌ها هنوز داده‌ای ندارند؛ بارگذاری و تحلیل کارایی باقی مانده‌اند.</p>

</div>


<div dir="rtl" lang="fa" style="text-align: right; line-height: 2;">

<h3>پایان کار: آزادسازی منابع دیتابیس</h3>
<p>این سلول را پس از پایان بخش‌های SQL اجرا کنید. اتصال‌های هر بلوک باید پیش‌تر با خروج از <code dir="ltr">with</code> آزاد شده باشند؛ اینجا pool متعلق به Engine آزاد می‌شود.</p>

</div>


In [31]:
if "engine" in globals():
    engine.dispose()
    print("Database engine pool disposed.")


Database engine pool disposed.
